E

In [2]:
import os
import hashlib
import numpy as np
import pandas as pd

FILE = 'genshin_impact.csv'

if not os.path.exists(FILE):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        raise FileNotFoundError('Letakkan genshin_impact.csv di folder yang sama dengan notebook ini.')

df = pd.read_csv(FILE)
df['release_date'] = pd.to_datetime(df['release_date'])
print('Ukuran dataset:', df.shape)

Saving genshin_impact.csv to genshin_impact.csv
Ukuran dataset: (103, 32)


In [3]:
# TODO E0 — Definisi dictionary PENANDAAN
PENANDAAN = {
    'Unnamed: 0': ('pengenal_langsung', 'hapus'),
    'name': ('pengenal_langsung', 'hash'),
    'rarity': ('sensitif', 'biarkan'),
    'weapon': ('quasi_identifier', 'biarkan'),
    'element': ('quasi_identifier', 'biarkan'),
    '3D-model': ('quasi_identifier', 'biarkan'),
    'region': ('quasi_identifier', 'biarkan'),
    'release_date': ('quasi_identifier', 'generalisasi'),
    'days_since_release': ('quasi_identifier', 'hapus'),
    'months_since_release': ('quasi_identifier', 'hapus'),
    'banner_versions': ('quasi_identifier', 'hapus'),
    'banner_dates': ('quasi_identifier', 'hapus'),
    'roles': ('sensitif', 'biarkan'),
    'constellation_0_pull': ('sensitif', 'biarkan'),
    'constellation_1_pull': ('sensitif', 'biarkan'),
    'constellation_2_pull': ('sensitif', 'biarkan'),
    'constellation_3_pull': ('sensitif', 'biarkan'),
    'constellation_4_pull': ('sensitif', 'biarkan'),
    'constellation_5_pull': ('sensitif', 'biarkan'),
    'constellation_6_pull': ('sensitif', 'biarkan'),
    'constellation_7_pull': ('sensitif', 'biarkan'),
    'asc_stat_bonus_category': ('quasi_identifier', 'biarkan'),
    'lvl_90_HP': ('quasi_identifier', 'hapus'),
    'lvl_90_ATK': ('quasi_identifier', 'hapus'),
    'lvl_90_DEF': ('quasi_identifier', 'hapus'),
    'duplicate_rate': ('sensitif', 'biarkan'),
    'c6_rate': ('sensitif', 'biarkan'),
    'avg_copies_per_player': ('sensitif', 'biarkan'),
    'num_banners': ('quasi_identifier', 'biarkan'),
    'is_standard_banner': ('quasi_identifier', 'biarkan'),
    'is_archon': ('quasi_identifier', 'hapus'),
    'pulled_count': ('sensitif', 'biarkan'),
}

In [4]:
# TODO E1 — HAPUS
kolom_hapus = [col for col, (kat, tind) in PENANDAAN.items() if tind == 'hapus' and col in df.columns]
aman = df.drop(columns=kolom_hapus).copy()

print(f"Jumlah kolom dibuang : {len(kolom_hapus)}")
print(f"Jumlah kolom tersisa : {aman.shape[1]}")

Jumlah kolom dibuang : 9
Jumlah kolom tersisa : 23


In [5]:
# TODO E2 — HASH
def samarkan(nilai, garam):
    if pd.isnull(nilai):
        return np.nan
    gabungan = str(garam) + str(nilai)
    hash_obj = hashlib.sha256(gabungan.encode('utf-8'))
    return hash_obj.hexdigest()[:12]

GARAM = 'rahasiatim_2025_eda_kmt'

kolom_hash = [col for col, (kat, tind) in PENANDAAN.items() if tind == 'hash']
for col in kolom_hash:
    if col in aman.columns:
        aman[col] = aman[col].apply(lambda x: samarkan(x, GARAM))

print(aman[['name', 'element', 'weapon', 'region']].head())

           name  element    weapon     region
0  c6f281db386f    Anemo     Sword   Fontaine
1  b5813abc687d  Electro  Catalyst     Natlan
2  70526d434ba6     Cryo  Claymore      Liyue
3  14fdb506f4cb   Dendro       Bow     Sumeru
4  dc6824714cde    Anemo     Sword  Mondstadt


In [6]:
# TODO E3 — Buktikan hash bersifat konsisten dan tidak terbaca
# a. Hash dua kali nilai yang sama, tunjukkan hasilnya identik
hash_1 = samarkan('lynette', GARAM)
hash_2 = samarkan('lynette', GARAM)
print(f"Hasil hash pertama ('lynette') : {hash_1}")
print(f"Hasil hash kedua   ('lynette') : {hash_2}")
print(f"Konsisten/Identik?             : {hash_1 == hash_2}")
print()

# b. Hash dua nilai berbeda, tunjukkan hasilnya berbeda
hash_3 = samarkan('varesa', GARAM)
print(f"Hasil hash nilai lain ('varesa'): {hash_3}")
print(f"Berbeda?                        : {hash_1 != hash_3}")

Hasil hash pertama ('lynette') : c6f281db386f
Hasil hash kedua   ('lynette') : c6f281db386f
Konsisten/Identik?             : True

Hasil hash nilai lain ('varesa'): b5813abc687d
Berbeda?                        : True


In [7]:
# TODO E4 — GENERALISASI
ACUAN = pd.Timestamp('2026-01-01')

# a. Hitung usia rilis karakter (dalam tahun) pada tanggal acuan 1 Januari 2026
usia_rilis = (ACUAN - df['release_date']).dt.days / 365.25

# b. Buat kolom 'kelompok_umur_rilis' dengan batas: <1 Thn, 1-2 Thn, 2-3 Thn, 3-4 Thn, 4+ Thn
bins = [0, 1, 2, 3, 4, 10]
labels = ['<1 Tahun', '1-2 Tahun', '2-3 Tahun', '3-4 Tahun', '4+ Tahun']
aman['kelompok_umur_rilis'] = pd.cut(usia_rilis, bins=bins, labels=labels)

# c. Buang kolom release_date dari 'aman'
aman = aman.drop(columns=['release_date'], errors='ignore')

# d. Rapikan kolom region dengan Title Case
aman['region'] = aman['region'].str.title()

print("Jumlah karakter per kelompok umur rilis:")
print(aman['kelompok_umur_rilis'].value_counts())
print("\nJumlah region unik:", aman['region'].nunique())
print(aman['region'].unique())

Jumlah karakter per kelompok umur rilis:
kelompok_umur_rilis
4+ Tahun     39
<1 Tahun     17
3-4 Tahun    17
2-3 Tahun    15
1-2 Tahun    15
Name: count, dtype: int64

Jumlah region unik: 9
['Fontaine' 'Natlan' 'Liyue' 'Sumeru' 'Mondstadt' 'Inazuma' 'Nod-Krai'
 'Snezhnaya' 'Extra-Terresterial']


F

In [8]:
# TODO F1 — tulis fungsi cek_k(data, kolom)
def cek_k(data, kolom):
    grup = data.groupby(kolom, observed=True).size()
    k_value = grup.min()
    grup_1_orang = (grup == 1).sum()
    total_grup = len(grup)
    return pd.Series({
        'k': k_value,
        'grup_1_karakter': grup_1_orang,
        'total_grup': total_grup
    })

In [9]:
# TODO F2 — ukur risiko SEBELUM penanganan
kolom_risiko_sebelum = ['region', '3D-model', 'weapon', 'release_date']
print(cek_k(df, kolom_risiko_sebelum))

k                    1
grup_1_karakter    100
total_grup         101
dtype: int64


In [10]:
# TODO F3 — ukur risiko SESUDAH generalisasi
kolom_risiko_sesudah = ['region', '3D-model', 'weapon', 'kelompok_umur_rilis']
print(cek_k(aman, kolom_risiko_sesudah))

k                   1
grup_1_karakter    80
total_grup         91
dtype: int64


In [11]:
# TODO F4 — TANTANGAN: capai k >= 5
aman['gender'] = aman['3D-model'].apply(lambda x: 'Female' if 'Female' in str(x) else 'Male')

print("Uji 1 — Quasi-identifier ['gender', 'weapon']:")
print(cek_k(aman, ['gender', 'weapon']))

print("\nUji 2 — Quasi-identifier ['3D-model']:")
print(cek_k(aman, ['3D-model']))

Uji 1 — Quasi-identifier ['gender', 'weapon']:
k                   6
grup_1_karakter     0
total_grup         10
dtype: int64

Uji 2 — Quasi-identifier ['3D-model']:
k                  11
grup_1_karakter     0
total_grup          5
dtype: int64


In [12]:
# TODO F5 — sel kecil pada tabel agregat
# a. Buat tabel silang jumlah karakter menurut region x element
tabel_silang = pd.crosstab(aman['region'], aman['element'])
print("Tabel Silang (Region x Element):")
display(tabel_silang)

# b. Hitung berapa sel yang terisi tetapi berisi kurang dari 5 karakter
sel_kecil = ((tabel_silang > 0) & (tabel_silang < 5)).sum().sum()
print(f"\nJumlah sel terisi yang berisi < 5 karakter: {sel_kecil}")

Tabel Silang (Region x Element):


element,Anemo,Cryo,Dendro,Electro,Geo,Hydro,Pyro
region,,,,,,,
Extra-Terresterial,0,1,0,0,0,0,0
Fontaine,1,4,1,1,1,3,2
Inazuma,4,1,1,4,3,2,2
Liyue,3,4,2,2,3,2,5
Mondstadt,3,4,0,2,2,3,4
Natlan,2,1,1,3,2,1,1
Nod-Krai,1,0,2,2,0,1,0
Snezhnaya,0,0,0,0,0,1,1
Sumeru,2,1,5,3,0,2,1



Jumlah sel terisi yang berisi < 5 karakter: 45


In [13]:
aman.to_csv('genshin_impact_clean.csv', index=False)

from google.colab import files
files.download('genshin_impact_clean.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>